In [11]:
import pandas as pd

# Load cleaned CSV file
file_name = "Cleaned_Sample_Dataset.csv"
df = pd.read_csv(file_name)

# Clean column names
df.columns = df.columns.str.strip()

# Convert date column
if "Order_Date" in df.columns:
    df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

# Convert numeric columns
numeric_columns = [
    "Quantity",
    "Unit_Price",
    "Sales",
    "Profit"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

# Basic Information
basic_information = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Duplicate Rows",
        "Missing Values"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df.isnull().sum().sum()
    ]
})

# Data Quality
data_quality = pd.DataFrame({
    "Column": df.columns,
    "Data Type": [str(df[column].dtype) for column in df.columns],
    "Missing Values": [df[column].isnull().sum() for column in df.columns],
    "Unique Values": [df[column].nunique() for column in df.columns]
})

# KPI Summary
kpi_data = {}

if "Sales" in df.columns:
    kpi_data["Total Sales"] = df["Sales"].sum()

if "Profit" in df.columns:
    kpi_data["Total Profit"] = df["Profit"].sum()

if "Quantity" in df.columns:
    kpi_data["Total Quantity"] = df["Quantity"].sum()

if "Order_ID" in df.columns:
    kpi_data["Total Orders"] = df["Order_ID"].nunique()

if "Sales" in df.columns:
    kpi_data["Average Sales"] = df["Sales"].mean()

kpi_summary = pd.DataFrame(
    list(kpi_data.items()),
    columns=["KPI", "Value"]
)

# Statistical Summary
statistics = df.describe(include="all").transpose().reset_index()
statistics = statistics.rename(columns={"index": "Column"})

# Category Analysis
if "Category" in df.columns:
    category_analysis = df.groupby("Category").agg(
        Orders=("Order_ID", "nunique") if "Order_ID" in df.columns else ("Category", "count"),
        Quantity=("Quantity", "sum") if "Quantity" in df.columns else ("Category", "count"),
        Sales=("Sales", "sum") if "Sales" in df.columns else ("Category", "count"),
        Profit=("Profit", "sum") if "Profit" in df.columns else ("Category", "count")
    ).reset_index()
else:
    category_analysis = pd.DataFrame()

# Product Analysis
if "Product" in df.columns:
    product_analysis = df.groupby("Product").agg(
        Quantity=("Quantity", "sum") if "Quantity" in df.columns else ("Product", "count"),
        Sales=("Sales", "sum") if "Sales" in df.columns else ("Product", "count"),
        Profit=("Profit", "sum") if "Profit" in df.columns else ("Product", "count")
    ).reset_index()
else:
    product_analysis = pd.DataFrame()

# City Analysis
if "City" in df.columns:
    city_analysis = df.groupby("City").agg(
        Orders=("Order_ID", "nunique") if "Order_ID" in df.columns else ("City", "count"),
        Sales=("Sales", "sum") if "Sales" in df.columns else ("City", "count"),
        Profit=("Profit", "sum") if "Profit" in df.columns else ("City", "count")
    ).reset_index()
else:
    city_analysis = pd.DataFrame()

# Order Status Analysis
if "Order_Status" in df.columns:
    order_status = df["Order_Status"].value_counts().reset_index()
    order_status.columns = ["Order_Status", "Order_Count"]
else:
    order_status = pd.DataFrame()

# Monthly Analysis
if "Order_Date" in df.columns and "Sales" in df.columns:
    monthly_analysis = df.dropna(subset=["Order_Date"]).copy()
    monthly_analysis["Month"] = monthly_analysis["Order_Date"].dt.to_period("M").astype(str)

    monthly_analysis = monthly_analysis.groupby("Month").agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum") if "Profit" in df.columns else ("Sales", "sum"),
        Quantity=("Quantity", "sum") if "Quantity" in df.columns else ("Sales", "count")
    ).reset_index()
else:
    monthly_analysis = pd.DataFrame()

# Top 5 Products
if not product_analysis.empty and "Sales" in product_analysis.columns:
    top_5_products = product_analysis.sort_values(
        "Sales", ascending=False
    ).head(5)
else:
    top_5_products = pd.DataFrame()

# Top 5 Cities
if not city_analysis.empty and "Sales" in city_analysis.columns:
    top_5_cities = city_analysis.sort_values(
        "Sales", ascending=False
    ).head(5)
else:
    top_5_cities = pd.DataFrame()

# Save all analysis results to Excel
output_file = "Analysis_Results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Cleaned_Data", index=False)
    basic_information.to_excel(writer, sheet_name="Basic_Information", index=False)
    data_quality.to_excel(writer, sheet_name="Data_Quality", index=False)
    kpi_summary.to_excel(writer, sheet_name="KPI_Summary", index=False)
    statistics.to_excel(writer, sheet_name="Statistics", index=False)
    category_analysis.to_excel(writer, sheet_name="Category_Analysis", index=False)
    product_analysis.to_excel(writer, sheet_name="Product_Analysis", index=False)
    city_analysis.to_excel(writer, sheet_name="City_Analysis", index=False)
    order_status.to_excel(writer, sheet_name="Order_Status", index=False)
    monthly_analysis.to_excel(writer, sheet_name="Monthly_Analysis", index=False)
    top_5_products.to_excel(writer, sheet_name="Top_5_Products", index=False)
    top_5_cities.to_excel(writer, sheet_name="Top_5_Cities", index=False)

print("Analysis completed successfully.")
print("Output file:", output_file)
print("Rows analyzed:", len(df))
print("Columns analyzed:", len(df.columns))

Analysis completed successfully.
Output file: Analysis_Results.xlsx
Rows analyzed: 30
Columns analyzed: 11
